## Recurrent Layers
## RNN Base

**class torch.nn.RNNBase(mode, input_size, hidden_size, num_layers=1, bias=True, batch_first=False, dropout=0.0, bidirectional=False, proj_size=0, device=None, dtype=None)**

Base class for RNN modules (RNN, LSTM, GRU)

Implements aspects of RNNs shared by the RNN, LSTM, and GRU classes, such as module initialization and utility methods for parameter storage management.

## RNN

** class torch.nn.RNN(input_size, hidden_size, num_layers=1, nonlinearity='tanh', bias=True, batch_first=False, dropout=0.0, bidirectional=False, device=None, dtype=None)**

Apply a multi-layer Ellman RNN with tanh or ReLU non-linerity to an input sequence. FOr each element inthe input sequence , ecah layer computes the following fun:
$$
h_t = \tanh\left(x_t W_{ih}^\top + b_{ih} + h_{t-1} W_{hh}^\top + b_{hh}\right)
$$

where htht​ is the hidden state at time t, xtxt​ is the input at time t, and h(t−1)h(t−1)​ is the hidden state of the previous layer at time t-1 or the initial hidden state at time 0. If nonlinearity is 'relu', then ReLUReLU is used instead of tanh⁡tanh.



In [ ]:
# Efficient implementation equivalent to the following with bidirectional=False
rnn = nn.RNN(input_size, hidden_size, num_layers)
params = dict(rnn.named_parameters())
def forward(x, hx=None, batch_first=False):
    if batch_first:
        x = x.transpose(0, 1)
    seq_len, batch_size, _ = x.size()
    if hx is None:
        hx = torch.zeros(rnn.num_layers, batch_size, rnn.hidden_size)
    h_t_minus_1 = hx.clone()
    h_t = hx.clone()
    output = []
    for t in range(seq_len):
        for layer in range(rnn.num_layers):
            input_t = x[t] if layer == 0 else h_t[layer - 1]
            h_t[layer] = torch.tanh(
                input_t @ params[f"weight_ih_l{layer}"].T
                + h_t_minus_1[layer] @ params[f"weight_hh_l{layer}"].T
                + params[f"bias_hh_l{layer}"]
                + params[f"bias_ih_l{layer}"]
            )
        output.append(h_t[-1].clone())
        h_t_minus_1 = h_t.clone()
    output = torch.stack(output)
    if batch_first:
        output = output.transpose(0, 1)
    return output, h_t

In [ ]:
import torch
import torch.nn as nn

rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=False)
params = dict(rnn.named_parameters())

def forward(x, hx=None, batch_first=False):
    if batch_first:
        x = x.transpose(0, 1)  # (seq_len, batch, input)

    seq_len, batch_size, _ = x.size()
    device = x.device
    dtype = x.dtype

    if hx is None:
        hx = torch.zeros(
            rnn.num_layers,
            batch_size,
            rnn.hidden_size,
            device=device,
            dtype=dtype,
        )

    h_t = hx.clone()
    output = []

    for t in range(seq_len):
        h_prev = h_t.clone()

        for layer in range(rnn.num_layers):
            input_t = x[t] if layer == 0 else h_t[layer - 1]

            h_t[layer] = torch.tanh(
                input_t @ params[f"weight_ih_l{layer}"].T
                + h_prev[layer] @ params[f"weight_hh_l{layer}"].T
                + params[f"bias_ih_l{layer}"]
                + params[f"bias_hh_l{layer}"]
            )

        output.append(h_t[-1].clone())

    output = torch.stack(output)  # (seq_len, batch, hidden)

    if batch_first:
        output = output.transpose(0, 1)

    return output, h_t


## LSTM
**class torch.LSTM(input_size, hidden_size, num_layers=1, bias=True, batch_first=False, dropout=0.0, bidirectional=False, proj_size=0, device=None, dtype=None)
Apply a multi-layer long short-term memory (LSTM) RNN to an input seq:. For each element in the input sequence, each layer computes the following function.
\begin{align}
i_t &= \sigma\left(W_{ii} x_t + b_{ii} + W_{hi} h_{t-1} + b_{hi}\right) \\
f_t &= \sigma\left(W_{if} x_t + b_{if} + W_{hf} h_{t-1} + b_{hf}\right) \\
g_t &= \tanh\left(W_{ig} x_t + b_{ig} + W_{hg} h_{t-1} + b_{hg}\right) \\
o_t &= \sigma\left(W_{io} x_t + b_{io} + W_{ho} h_{t-1} + b_{ho}\right) \\
c_t &= f_t \odot c_{t-1} + i_t \odot g_t \\
h_t &= o_t \odot \tanh(c_t)
\end{align}
​

where htht​ is the hidden state at time t, ctct​ is the cell state at time t, xtxt​ is the input at time t, ht−1ht−1​ is the hidden state of the layer at time t-1 or the initial hidden state at time 0, and itit​, ftft​, gtgt​, otot​ are the input, forget, cell, and output gates, respectively. σσ is the sigmoid function, and ⊙⊙ is the Hadamard product.

In a multilayer LSTM the input $$ X_t^{\,l} $$ of the l-th layer )l>=2 ) is the hiddent state $$ h_t^{\,l-1} $$ of the previous layer multiplied by dropout $$ δ_t^{\, l-1}  $$ where each $$ δ_t^{\, l-1}  $$  is a Bernauli random variable which is 0 with probability `dropout`.


If proj_size > 0 is specified, LSTM with projections will be used. This changes the LSTM cell in the following way. First, the dimension of ht will be changed from hidden_size to proj_size (dimensions of WhiWhi​ will be changed accordingly). Second, the output hidden state of each layer will be multiplied by a learnable projection matrix: $$ h_t=W_h_r h_t ​ = W_h_r ​_ht $$​. Note that as a consequence of this, the output of LSTM network will be of different shape as well

* input_size – The number of expected features in the input x
* hidden_size – The number of features in the hidden state h
* num_layers – Number of recurrent layers. E.g., setting num_layers=2 would mean stacking two LSTMs together to form a stacked LSTM, with the second LSTM taking in outputs of the first LSTM and computing the final results. Default: 1
* bias – If False, then the layer does not use bias weights b_ih and b_hh. Default: True

* batch_first – If True, then the input and output tensors are provided as (batch, seq, feature) instead of (seq, batch, feature). Note that this does not apply to hidden or cell states. See the Inputs/Outputs sections below for details. Default: False
* dropout – If non-zero, introduces a Dropout layer on the outputs of each LSTM layer except the last layer, with dropout probability equal to dropout. Default: 0

* bidirectional – If True, becomes a bidirectional LSTM. Default: False
* proj_size – If > 0, will use LSTM with projections of corresponding size. Default: 0


In [3]:
rnn = nn.LSTM(10, 20, 2)
input = torch.randn(5, 3, 10)
h0 = torch.randn(2, 3, 20)
c0 = torch.randn(2, 3, 20)
output, (hn, cn) = rnn(input, (h0, c0))

## GRU
class torch.nn.GRU(input_size, hidden_size, num_layers=1, bias=True, batch_first=False, dropout=0.0, bidirectional=False, device=None, dtype=None)

Apply a  multi-layer gated reurrent unit (GRU) RNN ot an input sequence. For each element in the input sequence, each layer computes the following fun:

\begin{align}
r_t &= \sigma\left(W_{ir} x_t + b_{ir} + W_{hr} h_{t-1} + b_{hr}\right) \\
z_t &= \sigma\left(W_{iz} x_t + b_{iz} + W_{hz} h_{t-1} + b_{hz}\right) \\
n_t &= \tanh\left(W_{in} x_t + b_{in}
      + r_t \odot \left(W_{hn} h_{t-1} + b_{hn}\right)\right) \\
h_t &= (1 - z_t) \odot n_t + z_t \odot h_{t-1}
\end{align}

where ht is the hidden state at time t, xt​ is the input at time t, h(t−1)​ is the hidden state of the layer at time t-1 or the initial hidden state at time 0, and rt​, zt​, nt​ are the reset, update, and new gates, respectively. σ is the sigmoid function, and ⊙ is the Hadamard product.


In [4]:
rnn = nn.GRU(10,  20, 2)
input = torch.randn(5, 3, 10)
h0 = torch.randn(2, 3, 20)
output, hn = rnn(input, h0)

## RNNCell

class torch.nn.RNNCell(input_size, hidden_size, bias=True, nonlineraity="tanh", device=None, dtype=None)

An  Elman RNN cell with tanh or ReLU non-linearity.

If nonlinearity is relu`, then ReLU is used in place of tanh.

###parameters:
* input_size (int) : The number of expected features in the input x
* hidden_size (int) The number of features in the hidden state h
* bias (bool) : If `False ` then the layer does not use the bias weights b_ih and b_hh. Defualt `True`
* nonlinearity (str): The non-linearity to use. Can be either `tanh` or `relu` Default `tanh`
### Inputs : inupt, hidden
* input : tensor contiaing input features
* hidden : tensor containing the initial hidden state Defaults to 0 if not provided

### Outputs : 'h'
* 'h' of shape (batch, batch_size) :tensor containing the next hidden state for each element in the batch
### Shape
* input $$ (N, H_in) or (H_in) $$
tensor contianing input features where Hin = Input_size
* hidden( N, Hout) or (Hout) tensor containing the initial hidden state where Hout = hidden_size
Defaults to zero
* output(N, Hout) or Hout tensor containing the next hidden state


* weight_ih (torch.Tensor) – the learnable input-hidden weights, of shape (hidden_size, input_size)
* weight_hh (torch.Tensor) – the learnable hidden-hidden weights, of shape (hidden_size, hidden_size)

* bias_ih = the learnable input-hidden bias, of shape (hidden_size)
* bias_hh - the learnable hidden-hidden bias, of shape (hidden_size)



In [5]:
rnn = nn.RNNCell(10, 20)
input = torch.randn(6, 3, 10)
hx = torch.randn(3, 20)
output = []
for i in range(6):
  hx = rnn(input[i], hx)
  output.append(hx)

## LSTMCell
class torch.nn.LSTMCell(input_size, hidden_size, bias=True, device=None, dtype=None)

A long short-term memory (LSTM) cell.
\begin{align}
i &= \sigma\left(W_{ii} x + b_{ii} + W_{hi} h + b_{hi}\right) \\
f &= \sigma\left(W_{if} x + b_{if} + W_{hf} h + b_{hf}\right) \\
g &= \tanh\left(W_{ig} x + b_{ig} + W_{hg} h + b_{hg}\right) \\
o &= \sigma\left(W_{io} x + b_{io} + W_{ho} h + b_{ho}\right) \\
c' &= f \odot c + i \odot g \\
h' &= o \odot \tanh(c')
\end{align}

where σσ is the sigmoid function, and ⊙⊙ is the Hadamard product.




On certain ROCm devices , when using float16 this module will use different prcision for backward.

### Examples:

In [7]:
rnn = nn.LSTMCell(10, 20) # (input_size, hidden_size)
input = torch.randn(2, 3, 10)  # (time_steps, batch, input_size)
hx = torch.randn(3, 20 )  # (batch, hidden_size)
cx = torch.randn(3, 20 )
output = []
for i in range(input.size()[0]):
  hx, cx  = rnn(input[i], (hx, cx))
  output.append(hx)
output = torch.stack(output, dim=0)


## GRUCell
class torch.nn.GRUCell(input_size, hidden_size, bias=True, device=None, dtype=None)

A gated recurrent unit (GRU) cell.
\begin{align}
r &= \sigma\left(W_{ir} x + b_{ir} + W_{hr} h + b_{hr}\right) \\
z &= \sigma\left(W_{iz} x + b_{iz} + W_{hz} h + b_{hz}\right) \\
n &= \tanh\left(W_{in} x + b_{in}
      + r \odot \left(W_{hn} h + b_{hn}\right)\right) \\
h' &= (1 - z) \odot n + z \odot h
\end{align}


In [11]:
rnn = nn.GRUCell(10, 20 )
input = torch.randn(6, 3, 10)
hx = torch.randn(3, 20)

outupt = []
for i in range(6):
  hx = rnn(input[i], hx)
  output.append(hx)